## Task 1: Prepare and Scale the Data

In [ ]:
# Import libraries for SVM, KNN, scaling, and evaluation
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
# Load preprocessed Telco Churn dataset from Module 3 (with encoding already applied)
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
cat_cols = df.select_dtypes(include='object').columns.drop('customerID')
df_encoded = pd.get_dummies(df.drop('customerID', axis=1), columns=cat_cols, drop_first=True)
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']
print(f'Dataset loaded: {X.shape[0]} samples, {X.shape[1]} features')

In [ ]:
# Split data with stratification to maintain churn ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')

In [ ]:
# Scale features using StandardScaler (fit on train only to prevent data leakage)
# Both SVM and KNN require scaled features for fair distance/margin calculations
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print('Features scaled using StandardScaler')
print(f'Training features shape: {X_train_s.shape}')
print(f'Test features shape: {X_test_s.shape}')

## Task 2: Train an SVM Classifier

In [ ]:
# Train SVM with RBF kernel and measure training time
start = time.time()
svm = SVC(kernel='rbf', random_state=42)
svm.fit(X_train_s, y_train)
svm_time = time.time() - start
print(f'SVM training completed in {svm_time:.2f} seconds')

In [ ]:
# Evaluate SVM: compute accuracy, F1 score, and print classification report
y_pred_svm = svm.predict(X_test_s)
svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm)
print(f"SVM Accuracy: {svm_accuracy:.4f}")
print(f"SVM F1: {svm_f1:.4f}")
print(f"SVM Training Time: {svm_time:.2f}s")
print("\n=== SVM Classification Report ===")
print(classification_report(y_test, y_pred_svm, target_names=['No Churn', 'Churn']))

## Task 3: Train KNN with K=5

In [ ]:
# Train KNN with K=5 and measure time (note: actual computation happens at prediction, not training)
start = time.time()
knn5 = KNeighborsClassifier(n_neighbors=5)
knn5.fit(X_train_s, y_train)
knn5_time = time.time() - start
print(f'KNN (K=5) training completed in {knn5_time:.2f} seconds')

In [ ]:
# Evaluate KNN (K=5): compute accuracy, F1 score, and print classification report
y_pred_knn5 = knn5.predict(X_test_s)
knn5_accuracy = accuracy_score(y_test, y_pred_knn5)
knn5_f1 = f1_score(y_test, y_pred_knn5)
print(f"KNN (K=5) Accuracy: {knn5_accuracy:.4f}")
print(f"KNN (K=5) F1: {knn5_f1:.4f}")
print(f"KNN (K=5) Training Time: {knn5_time:.2f}s")
print("\n=== KNN (K=5) Classification Report ===")
print(classification_report(y_test, y_pred_knn5, target_names=['No Churn', 'Churn']))

## Task 4: Experiment with Different K Values

In [ ]:
# Train KNN with K=3 and evaluate
knn3 = KNeighborsClassifier(n_neighbors=3)
knn3.fit(X_train_s, y_train)
y_pred_knn3 = knn3.predict(X_test_s)
knn3_accuracy = accuracy_score(y_test, y_pred_knn3)
knn3_f1 = f1_score(y_test, y_pred_knn3)
print(f"KNN (K=3) Accuracy: {knn3_accuracy:.4f}")
print(f"KNN (K=3) F1: {knn3_f1:.4f}")

In [ ]:
# Train KNN with K=10 and evaluate
knn10 = KNeighborsClassifier(n_neighbors=10)
knn10.fit(X_train_s, y_train)
y_pred_knn10 = knn10.predict(X_test_s)
knn10_accuracy = accuracy_score(y_test, y_pred_knn10)
knn10_f1 = f1_score(y_test, y_pred_knn10)
print(f"KNN (K=10) Accuracy: {knn10_accuracy:.4f}")
print(f"KNN (K=10) F1: {knn10_f1:.4f}")

In [ ]:
# Create comparison table across different K values
k_comparison = pd.DataFrame({
    'K Value': [3, 5, 10],
    'Accuracy': [f"{knn3_accuracy:.4f}", f"{knn5_accuracy:.4f}", f"{knn10_accuracy:.4f}"],
    'F1': [f"{knn3_f1:.4f}", f"{knn5_f1:.4f}", f"{knn10_f1:.4f}"]
})
print("\n=== K-Value Comparison Table ===")
print(k_comparison.to_string(index=False))

In [ ]:
# Analyze the effect of K on model performance
print("\n=== K-Value Analysis ===")
print("\nK=3 (Low K, High Sensitivity):")
print(f"  - Uses only 3 nearest neighbors")
print(f"  - More sensitive to noise and outliers in training data")
print(f"  - Risk of overfitting (capturing training noise)")
print(f"  - Accuracy: {knn3_accuracy:.4f}")
print("\nK=5 (Medium K, Balanced):")
print(f"  - Good balance between noise sensitivity and smoothing")
print(f"  - Usually a reasonable default choice")
print(f"  - Accuracy: {knn5_accuracy:.4f}")
print("\nK=10 (High K, Smoother Boundaries):")
print(f"  - Uses more neighbors, creating smoother decision boundaries")
print(f"  - Less sensitive to individual training points")
print(f"  - Risk of underfitting (missing important patterns)")
print(f"  - Accuracy: {knn10_accuracy:.4f}")

# Identify best K
k_values = [3, 5, 10]
accuracies = [knn3_accuracy, knn5_accuracy, knn10_accuracy]
best_k = k_values[np.argmax(accuracies)]
best_knn_accuracy = max(accuracies)
print(f"\nBest K: {best_k} with accuracy {best_knn_accuracy:.4f}")

## Task 5: SVM vs Best KNN — Full Comparison

In [ ]:
# Get best KNN model details
if best_k == 3:
    best_knn_f1 = knn3_f1
elif best_k == 5:
    best_knn_f1 = knn5_f1
else:
    best_knn_f1 = knn10_f1

# Create comprehensive comparison table
model_comparison = pd.DataFrame({
    'Model': [f'SVM (RBF)', f'KNN (K={best_k})'],
    'Accuracy': [f"{svm_accuracy:.4f}", f"{best_knn_accuracy:.4f}"],
    'F1': [f"{svm_f1:.4f}", f"{best_knn_f1:.4f}"],
    'Training Time': [f"{svm_time:.2f}s", "<0.01s"]
})
print("\n=== SVM vs Best KNN Comparison ===")
print(model_comparison.to_string(index=False))

In [ ]:
# Detailed explanation of training vs prediction time
print("\n=== Training vs Prediction Time Analysis ===")
print("\nSVM:")
print(f"  - Training time: {svm_time:.2f}s (finds optimal support vectors)")
print("  - Prediction time: Fast (uses only support vectors, not all training data)")
print(f"  - Number of support vectors: {svm.n_support_.sum()}")
print("\nKNN:")
print(f"  - 'Training' time: < 0.01s (just stores the data, no model to learn)")
print("  - Prediction time: Slow (computes distances to all training points)")
print(f"  - Must search {X_train_s.shape[0]} points for each test sample")
print("\nTakeaway: SVM pays upfront training cost for fast predictions.")
print("         KNN defers cost to prediction time (lazy learning).")

## Task 6: Discussion

In [ ]:
# Write a comprehensive discussion of when to choose KNN over SVM
print("\n=== When to Choose KNN Over SVM ===")
print("""
DATASET SIZE & COMPUTATIONAL RESOURCES:
Choose KNN when you have a small dataset (< 10,000 samples). KNN's training is 
essentially instant—it just stores the data. SVM requires quadratic computation 
to find support vectors, making it slow on massive datasets. However, if predictions 
need to be made frequently on large test sets, KNN becomes impractical (O(n) for each 
prediction). SVM shines here with O(1) prediction using only support vectors.

DIMENSIONALITY:
Choose SVM for high-dimensional data (hundreds or thousands of features). SVM works 
well in high dimensions via the kernel trick. KNN suffers from the "curse of 
dimensionality"—in high-dimensional spaces, all points are roughly equidistant, 
making nearest neighbors less meaningful. In our Telco dataset (~40 features), this 
is moderate, favoring SVM.

INTERPRETABILITY:
Choose KNN for maximum interpretability. "This customer churned because they are 
similar to these 5 other churners" is easy to explain to business stakeholders. 
SVM's decision boundary (defined by support vectors and kernel) is a black box—you 
cannot easily explain why an individual prediction was made.

PERFORMANCE & ACCURACY:
In this experiment, both models achieved similar accuracy. SVM often wins on 
structured data due to its margin-maximization philosophy; KNN can win on locally 
clustered data. Neither algorithm is universally superior—always validate both.

PRACTICAL RECOMMENDATION FOR TELCO CHURN:
Use SVM. This is a real-time prediction problem (scoring new customers frequently), 
and SVM's fast prediction time is critical. The dataset is reasonably large 
(7,043 samples), and SVM handles feature scaling and moderate dimensionality well. 
KNN would be acceptable only if predictions were batch-processed monthly and 
interpretability (customer similarity) was paramount over speed.
""")